# Tema 4 - Ecuaciones de Nudos y Mallas para Circuitos Resistivos

**Teoría de Circuitos - ETSI, Universidad de Sevilla**

---

## Objetivos de aprendizaje

- Comprender el balance de incógnitas y ecuaciones en un circuito resistivo
- Formular sistemáticamente las **ecuaciones de nudos** (análisis nodal) y escribir la matriz de conductancias $\mathbf{G}$
- Formular sistemáticamente las **ecuaciones de mallas** y escribir la matriz de resistencias $\mathbf{R}$
- Manejar **fuentes de tensión** en análisis nodal (supernudo) y **fuentes de corriente** en análisis de mallas (supermalla)
- Incorporar **fuentes dependientes** al sistema de ecuaciones
- Resolver ejercicios tipo boletín de la asignatura con ambos métodos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
import schemdraw
import schemdraw.elements as elm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA = '#cb181d'       # rojo - rectas, límites
COLOR_PUNTO = '#238b45'       # verde - puntos de operación, resultados
COLOR_N = '#a6cee3'           # azul claro
COLOR_P = '#b2df8a'           # verde claro

print('Configuración lista.')

---

## 1. Balance de incógnitas y ecuaciones

En un circuito con $n$ nudos y $c$ componentes, las incógnitas son las tensiones y corrientes de cada componente:

- **Incógnitas totales**: $2c$ (una tensión $v_k$ y una corriente $i_k$ por componente)
- **Ecuaciones disponibles**: $2c$

Las ecuaciones se desglosan en:

| Tipo de ecuación | Cantidad | Origen |
|------------------|----------|--------|
| Ecuaciones de componente (V-I) | $c$ | Ley de Ohm, relaciones de fuentes |
| Ecuaciones de KCL (nudos) | $n - 1$ | Ley de corrientes de Kirchhoff |
| Ecuaciones de KVL (mallas) | $c - n + 1$ | Ley de tensiones de Kirchhoff |
| **Total** | **$c + (n-1) + (c-n+1) = 2c$** | |

$$\boxed{2c \text{ incógnitas} = c \text{ (componente)} + (n-1) \text{ (KCL)} + (c-n+1) \text{ (KVL)}}$$

**Ejemplo**: Un circuito con $n = 3$ nudos y $c = 4$ componentes tiene:
- $2 \times 4 = 8$ incógnitas
- $4$ ecuaciones de componente + $2$ KCL + $2$ KVL $= 8$ ecuaciones

> **Clave**: El sistema siempre está determinado. Los métodos de nudos y mallas reducen el número de ecuaciones a resolver.

In [ ]:
# Circuito ejemplo: 3 nudos, 4 componentes
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_title('Ejemplo: circuito con 3 nudos y 4 componentes', fontsize=14, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)

# Nudo 1 (arriba izquierda) -> Nudo 2 (arriba derecha) via R1
d += (R1 := elm.Resistor().right().label(r'$R_1$'))
d += elm.Dot().label(r'$\mathbf{2}$', loc='right')

# Nudo 2 -> Nudo 3 (abajo derecha) via R2
d += (R2 := elm.Resistor().down().label(r'$R_2$', loc='left'))
d += elm.Dot().label(r'$\mathbf{3}$ (ref)', loc='right')

# Nudo 3 -> Nudo 1 (abajo izquierda -> arriba izquierda) via Ig
d += elm.SourceI().left().label(r'$I_g$').reverse()

# Nudo 1 arriba
d += elm.Resistor().up().label(r'$R_3$', loc='left')
d += elm.Dot().label(r'$\mathbf{1}$', loc='left')

# R4 entre nudo 1 y nudo 3 (diagonal = via nudo 2 rama)
d += elm.Line().right().at(R1.start)
d.here = R1.start
d += elm.Dot()

d.draw()
plt.tight_layout()
plt.show()

---

## 2. Ecuaciones de nudos (análisis nodal)

El método de nudos reduce el sistema a $n - 1$ ecuaciones con $n - 1$ incógnitas (las tensiones de nudo).

### 2.1 Procedimiento sistemático

1. **Elegir un nudo de referencia** (tierra, $V_{\text{ref}} = 0$). Se elige el que tenga más conexiones.
2. **Asignar tensiones** $V_1, V_2, \ldots, V_{n-1}$ a los demás nudos.
3. **Escribir KCL en cada nudo** $k$: la suma de corrientes que salen del nudo es cero.
4. **Sustituir** las corrientes por las tensiones usando la ley de Ohm: $I_{kj} = G_{kj}(V_k - V_j)$.
5. **Resolver** el sistema lineal.

### 2.2 Forma matricial

Para un circuito con solo resistencias y fuentes de corriente independientes:

$$\boxed{\mathbf{G} \cdot \mathbf{V} = \mathbf{I}_s}$$

donde:
- $\mathbf{G}$ es la **matriz de conductancias** ($n-1 \times n-1$)
- $\mathbf{V} = [V_1, V_2, \ldots, V_{n-1}]^T$ son las tensiones de nudo
- $\mathbf{I}_s$ es el vector de fuentes de corriente

### 2.3 Construcción de la matriz $\mathbf{G}$

| Elemento | Regla |
|----------|-------|
| $G_{kk}$ (diagonal) | Suma de **todas** las conductancias conectadas al nudo $k$ |
| $G_{kj}$ (fuera de diagonal) | $-$ (suma de conductancias entre nudos $k$ y $j$) |

$$G_{kk} = \sum_{j} G_{kj}^{\text{(conectadas a } k)} \qquad G_{kj} = -G_{jk}^{\text{(entre } k \text{ y } j)}$$

> **Propiedad**: La matriz $\mathbf{G}$ es **simétrica** cuando no hay fuentes dependientes: $G_{kj} = G_{jk}$.

### 2.4 Vector de fuentes $\mathbf{I}_s$

- Si una fuente de corriente $I_g$ **entra** al nudo $k$: se suma $+I_g$ en la fila $k$
- Si una fuente de corriente $I_g$ **sale** del nudo $k$: se suma $-I_g$ en la fila $k$

---

## 3. Ecuaciones de mallas (análisis de mallas)

El método de mallas reduce el sistema a $c - n + 1$ ecuaciones con $c - n + 1$ incógnitas (las corrientes de malla).

### 3.1 Procedimiento sistemático

1. **Identificar las mallas** independientes (ventanas del circuito).
2. **Asignar corrientes de malla** $I_1, I_2, \ldots, I_m$ en sentido horario (convención).
3. **Escribir KVL en cada malla** $k$: la suma de caídas de tensión en sentido horario es cero.
4. **Sustituir** las tensiones por corrientes usando la ley de Ohm: $V_R = R \cdot (I_k - I_j)$.
5. **Resolver** el sistema lineal.

### 3.2 Forma matricial

Para un circuito con solo resistencias y fuentes de tensión independientes:

$$\boxed{\mathbf{R} \cdot \mathbf{I} = \mathbf{V}_s}$$

donde:
- $\mathbf{R}$ es la **matriz de resistencias** ($m \times m$, con $m = c - n + 1$)
- $\mathbf{I} = [I_1, I_2, \ldots, I_m]^T$ son las corrientes de malla
- $\mathbf{V}_s$ es el vector de fuentes de tensión

### 3.3 Construcción de la matriz $\mathbf{R}$

| Elemento | Regla |
|----------|-------|
| $R_{kk}$ (diagonal) | Suma de **todas** las resistencias en la malla $k$ |
| $R_{kj}$ (fuera de diagonal) | $-$ (suma de resistencias compartidas entre mallas $k$ y $j$) |

> **Propiedad**: La matriz $\mathbf{R}$ es **simétrica** cuando no hay fuentes dependientes: $R_{kj} = R_{jk}$.

### 3.4 Vector de fuentes $\mathbf{V}_s$

- Si una fuente de tensión $E_g$ produce una subida en el sentido de recorrido de la malla $k$: se suma $+E_g$ en la fila $k$
- Si produce una caída: se suma $-E_g$

---

## 4. Fuentes de tensión en ecuaciones de nudos: el supernudo

Cuando hay una **fuente de tensión** $E_g$ entre dos nudos $k$ y $j$ (ninguno de referencia), no podemos escribir directamente la KCL porque no conocemos la corriente por la fuente.

### 4.1 Concepto de supernudo

Se agrupan los nudos $k$ y $j$ en un **supernudo**. Las ecuaciones son:

1. **Ecuación de restricción** (la fuente impone la diferencia de tensión):

$$\boxed{V_k - V_j = E_g}$$

2. **KCL del supernudo** (se escribe para la superficie que engloba ambos nudos):

$$\sum \text{corrientes que salen del supernudo} = 0$$

### 4.2 Caso especial: fuente conectada a referencia

Si la fuente $E_g$ está entre el nudo $k$ y la referencia:

$$V_k = E_g$$

La tensión del nudo es directamente conocida. Se elimina esa incógnita y se reduce el sistema.

> **Truco para el examen**: Antes de montar el sistema, mira si alguna fuente de tensión fija directamente una tensión de nudo. Eso simplifica mucho.

---

## 5. Fuentes de corriente en ecuaciones de mallas: la supermalla

Cuando hay una **fuente de corriente** $I_g$ en una rama compartida por las mallas $k$ y $j$, no podemos escribir la KVL directamente porque no conocemos la tensión en bornes de la fuente.

### 5.1 Concepto de supermalla

Se combinan las mallas $k$ y $j$ en una **supermalla** (se recorre el contorno exterior de ambas, saltando la fuente de corriente).

1. **Ecuación de restricción** (la fuente impone la diferencia de corrientes de malla):

$$\boxed{I_k - I_j = \pm I_g}$$

(el signo depende de la orientación)

2. **KVL de la supermalla** (se escribe recorriendo el contorno exterior).

### 5.2 Caso especial: fuente en rama exterior

Si la fuente de corriente $I_g$ está solo en la malla $k$ (rama exterior):

$$I_k = \pm I_g$$

La corriente de malla es directamente conocida. Se elimina esa incógnita.

---

## 6. Fuentes dependientes

Las fuentes dependientes (controladas) añaden una relación entre la variable de control y la fuente:

| Tipo | Símbolo | Relación |
|------|---------|----------|
| Fuente de tensión controlada por tensión (FTCT) | $E = \mu \cdot V_x$ | Tensión $\to$ Tensión |
| Fuente de tensión controlada por corriente (FTCI) | $E = r \cdot I_x$ | Corriente $\to$ Tensión |
| Fuente de corriente controlada por tensión (FICT) | $I = g \cdot V_x$ | Tensión $\to$ Corriente |
| Fuente de corriente controlada por corriente (FICC) | $I = \alpha \cdot I_x$ | Corriente $\to$ Corriente |

### 6.1 Procedimiento

1. Escribir las ecuaciones de nudos o mallas **como si la fuente dependiente fuera independiente**.
2. **Expresar la variable de control** ($V_x$ o $I_x$) en función de las incógnitas del sistema (tensiones de nudo o corrientes de malla).
3. **Sustituir** en las ecuaciones.
4. Reagrupar términos y resolver.

> **Importante**: Con fuentes dependientes, la matriz $\mathbf{G}$ (o $\mathbf{R}$) **ya no es simétrica** en general.

### 6.2 Error frecuente

No confundir la variable de control con una incógnita independiente. Siempre hay que expresarla en función de las $V_k$ o $I_k$ del sistema.

---

## 7. Ejercicios resueltos del boletín

A continuación se resuelven los problemas tipo del boletín de la asignatura, aplicando los métodos de nudos y mallas de forma sistemática.

### 7.1 Problema 1: Análisis nodal básico (3 nudos)

**Datos:** Circuito con 3 nudos ($A$, $B$, referencia). $I_g = 10$ A, $R_1 = 2\;\Omega$, $R_2 = 4\;\Omega$, $R_3 = 5\;\Omega$, $R_4 = 10\;\Omega$.

La fuente de corriente $I_g$ entra por el nudo $A$.

#### Ejercicio resuelto: Análisis nodal P1

**Paso 1:** Elegir el nudo inferior como referencia. Incógnitas: $V_A$, $V_B$.

**Paso 2:** Escribir KCL en cada nudo.

**Nudo A** (corrientes que salen):

$$G_1 \cdot V_A + G_2 \cdot (V_A - V_B) = I_g$$

$$\frac{1}{2} V_A + \frac{1}{4}(V_A - V_B) = 10$$

**Nudo B** (corrientes que salen):

$$G_2 \cdot (V_B - V_A) + G_3 \cdot V_B + G_4 \cdot V_B = 0$$

$$\frac{1}{4}(V_B - V_A) + \frac{1}{5} V_B + \frac{1}{10} V_B = 0$$

**Paso 3:** Forma matricial $\mathbf{G} \cdot \mathbf{V} = \mathbf{I}_s$:

$$\begin{pmatrix} \frac{1}{2} + \frac{1}{4} & -\frac{1}{4} \\ -\frac{1}{4} & \frac{1}{4} + \frac{1}{5} + \frac{1}{10} \end{pmatrix} \begin{pmatrix} V_A \\ V_B \end{pmatrix} = \begin{pmatrix} 10 \\ 0 \end{pmatrix}$$

$$\begin{pmatrix} 0.75 & -0.25 \\ -0.25 & 0.55 \end{pmatrix} \begin{pmatrix} V_A \\ V_B \end{pmatrix} = \begin{pmatrix} 10 \\ 0 \end{pmatrix}$$

**Paso 4:** Resolver el sistema.

$$\Delta = 0.75 \times 0.55 - (-0.25)^2 = 0.4125 - 0.0625 = 0.35$$

$$V_A = \frac{10 \times 0.55 - 0 \times (-0.25)}{0.35} = \frac{5.5}{0.35} = 15.71\;\text{V}$$

$$V_B = \frac{0.75 \times 0 - (-0.25) \times 10}{0.35} = \frac{2.5}{0.35} = 7.14\;\text{V}$$

**Paso 5:** Verificación. Corriente por $R_2$:

$$I_{R_2} = \frac{V_A - V_B}{R_2} = \frac{15.71 - 7.14}{4} = 2.14\;\text{A}$$

$$\boxed{V_A = 15.71\;\text{V}, \quad V_B = 7.14\;\text{V}}$$

In [ ]:
# Diagrama P1: Análisis nodal, 3 nudos
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_title('P1: Análisis nodal - 3 nudos', fontsize=14, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)

# Rama izquierda: Ig (de ref a A)
d += elm.SourceI().up().label(r'$I_g=10$ A').reverse()
d += elm.Dot().label(r'$A$', loc='left')
A = d.here

# Rama superior: R1 de A hacia derecha
d += elm.Resistor().right().label(r'$R_1=2\;\Omega$')

# De A hacia abajo: R3
d.here = A
d += elm.Resistor().right().length(3).label(r'$R_2=4\;\Omega$')
d += elm.Dot().label(r'$B$', loc='right')
B = d.here

# R3 de B hacia abajo
d += elm.Resistor().down().label(r'$R_3=5\;\Omega$', loc='right')
ref_right = d.here

# R4 de B a ref
d.here = B
d += elm.Resistor().down().at(B).label(r'$R_4=10\;\Omega$', loc='left')

# Conexión R1 a A y abajo
d.here = A
d += elm.Resistor().down().label(r'$R_1=2\;\Omega$', loc='left')
d += elm.Ground()

d.draw()
plt.tight_layout()
plt.show()

### 7.2 Problema 2: Análisis nodal con 5 nudos

**Datos:** Circuito con 5 nudos. $I_g = 5$ A entra al nudo $A$. $R_1 = R_2 = R_3 = R_4 = R_5 = 1\;\Omega$. Encontrar $V_A$, $V_B$, $V_C$.

#### Ejercicio resuelto: Análisis nodal P2

**Paso 1:** Nudo de referencia: nudo $D$ (tierra). Dos nudos adicionales $E$ quedan en el camino. Simplificando la topología a 3 incógnitas esenciales: $V_A$, $V_B$, $V_C$.

**Paso 2:** KCL en cada nudo (todas las $G_k = 1$ S):

**Nudo A:**
$$3V_A - V_B - V_C = 5$$

**Nudo B:**
$$-V_A + 3V_B - V_C = 0$$

**Nudo C:**
$$-V_A - V_B + 3V_C = 0$$

**Paso 3:** Forma matricial:

$$\begin{pmatrix} 3 & -1 & -1 \\ -1 & 3 & -1 \\ -1 & -1 & 3 \end{pmatrix} \begin{pmatrix} V_A \\ V_B \\ V_C \end{pmatrix} = \begin{pmatrix} 5 \\ 0 \\ 0 \end{pmatrix}$$

**Paso 4:** Resolver (por simetría $V_B = V_C$):

De las ecuaciones 2 y 3: $-V_A + 3V_B - V_B = 0 \implies V_B = V_C$.

Sustituyendo en ecuación 1: $3V_A - 2V_B = 5$.

De ecuación 2: $-V_A + 2V_B = 0 \implies V_A = 2V_B$.

$$3(2V_B) - 2V_B = 5 \implies 4V_B = 5 \implies V_B = 1.25\;\text{V}$$

$$\boxed{V_A = 2.50\;\text{V}, \quad V_B = V_C = 1.25\;\text{V}}$$

In [ ]:
# Verificación numérica P2
G = np.array([[3, -1, -1],
              [-1, 3, -1],
              [-1, -1, 3]], dtype=float)
Is = np.array([5, 0, 0], dtype=float)
V = np.linalg.solve(G, Is)

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_title('P2: Tensiones de nudo (verificación)', fontsize=14, fontweight='bold')
nudos = [r'$V_A$', r'$V_B$', r'$V_C$']
barras = ax.bar(nudos, V, color=[COLOR_PRINCIPAL, COLOR_RECTA, COLOR_PUNTO], width=0.5, edgecolor='black')
for bar, val in zip(barras, V):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f} V', ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel(r'Tensión (V)')
ax.set_ylim(0, max(V)*1.3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.3 Problema 3: Análisis nodal con fuente de tensión ($E_g$)

**Datos:** $E_g = 12$ V entre nudo $A$ y referencia, $R_1 = 3\;\Omega$, $R_2 = 6\;\Omega$, $R_3 = 2\;\Omega$, $I_g = 4$ A entra a nudo $B$.

#### Ejercicio resuelto: Análisis nodal P3

**Paso 1:** La fuente de tensión fija directamente $V_A$:

$$V_A = E_g = 12\;\text{V}$$

**Paso 2:** Solo queda una incógnita: $V_B$. KCL en nudo $B$:

$$\frac{V_B - V_A}{R_1} + \frac{V_B}{R_2} + \frac{V_B}{R_3} = -I_g$$

$$\frac{V_B - 12}{3} + \frac{V_B}{6} + \frac{V_B}{2} = -4$$

**Paso 3:** Resolver:

$$V_B \left(\frac{1}{3} + \frac{1}{6} + \frac{1}{2}\right) = -4 + \frac{12}{3}$$

$$V_B \cdot 1.0 = -4 + 4 = 0$$

$$\boxed{V_A = 12\;\text{V}, \quad V_B = 0\;\text{V}}$$

> **Observación**: El nudo $B$ queda a $0$ V (mismo potencial que la referencia) por la combinación particular de valores.

### 7.4 Problema 4: Supernudo (fuente de tensión entre dos nudos)

**Datos:** $E_g = 6$ V entre nudos $A$ y $B$ ($V_A - V_B = 6$ V). $R_1 = 2\;\Omega$ (de $A$ a ref), $R_2 = 4\;\Omega$ (de $B$ a ref), $R_3 = 3\;\Omega$ (de $A$ a $C$), $R_4 = 6\;\Omega$ (de $C$ a ref), $I_g = 3$ A entra a nudo $C$.

#### Ejercicio resuelto: Supernudo P4

**Paso 1:** Supernudo $\{A, B\}$. Ecuaciones:

**Restricción del supernudo:**
$$V_A - V_B = 6$$

**KCL del supernudo** (corrientes que salen de $A$ y $B$ juntos):

$$\frac{V_A}{R_1} + \frac{V_B}{R_2} + \frac{V_A - V_C}{R_3} = 0$$

$$\frac{V_A}{2} + \frac{V_B}{4} + \frac{V_A - V_C}{3} = 0$$

**KCL nudo C:**
$$\frac{V_C - V_A}{R_3} + \frac{V_C}{R_4} = -I_g$$

$$\frac{V_C - V_A}{3} + \frac{V_C}{6} = -3$$

**Paso 2:** Tres ecuaciones, tres incógnitas ($V_A$, $V_B$, $V_C$). Sustituimos $V_A = V_B + 6$:

De KCL supernudo:
$$\frac{V_B + 6}{2} + \frac{V_B}{4} + \frac{V_B + 6 - V_C}{3} = 0$$

$$V_B\left(\frac{1}{2} + \frac{1}{4} + \frac{1}{3}\right) + \frac{6}{2} + \frac{6}{3} - \frac{V_C}{3} = 0$$

$$\frac{13}{12}V_B - \frac{V_C}{3} = -5$$

De KCL nudo C:
$$\frac{V_C - V_B - 6}{3} + \frac{V_C}{6} = -3$$

$$-\frac{V_B}{3} + \frac{V_C}{2} = -3 + 2 = -1$$

**Paso 3:** Resolver el sistema $2 \times 2$:

De la segunda: $V_B = \frac{3}{2}V_C + 3$. Sustituyendo:

$$\frac{13}{12}\left(\frac{3V_C}{2} + 3\right) - \frac{V_C}{3} = -5$$

$$\frac{13V_C}{8} + \frac{13}{4} - \frac{V_C}{3} = -5$$

$$V_C\left(\frac{13}{8} - \frac{1}{3}\right) = -5 - 3.25 = -8.25$$

$$V_C \cdot \frac{31}{24} = -8.25 \implies V_C = \frac{-8.25 \times 24}{31} = -6.39\;\text{V}$$

$$V_B = \frac{3(-6.39)}{2} + 3 = -6.58\;\text{V}$$

$$V_A = V_B + 6 = -0.58\;\text{V}$$

$$\boxed{V_A = -0.58\;\text{V}, \quad V_B = -6.58\;\text{V}, \quad V_C = -6.39\;\text{V}}$$

In [ ]:
# Verificación numérica P4
# VA - VB = 6, (13/12)VB - VC/3 = -5, -VB/3 + VC/2 = -1
A = np.array([[1, -1, 0],
              [0, 13/12, -1/3],
              [0, -1/3, 1/2]], dtype=float)
b = np.array([6, -5, -1], dtype=float)
sol = np.linalg.solve(A, b)

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_title('P4: Tensiones de nudo con supernudo', fontsize=14, fontweight='bold')
nudos = [r'$V_A$', r'$V_B$', r'$V_C$']
colors = [COLOR_PRINCIPAL, COLOR_RECTA, COLOR_PUNTO]
barras = ax.bar(nudos, sol, color=colors, width=0.5, edgecolor='black')
for bar, val in zip(barras, sol):
    ypos = bar.get_height() + 0.1 if val >= 0 else bar.get_height() - 0.4
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'{val:.2f} V', ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel(r'Tensión (V)')
ax.axhline(y=0, color='black', lw=0.8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.5 Problema 6: Análisis de mallas (2 mallas)

**Datos:** Dos mallas. $E_1 = 10$ V, $E_2 = 5$ V, $R_1 = 2\;\Omega$, $R_2 = 3\;\Omega$, $R_3 = 4\;\Omega$ (compartida).

#### Ejercicio resuelto: Análisis de mallas P6

**Paso 1:** Asignar corrientes de malla $I_1$ (horario, malla izquierda), $I_2$ (horario, malla derecha).

**Paso 2:** KVL malla 1 (sentido $I_1$):

$$R_1 \cdot I_1 + R_3 \cdot (I_1 - I_2) = E_1$$

$$2I_1 + 4(I_1 - I_2) = 10$$

$$6I_1 - 4I_2 = 10$$

**KVL malla 2:**

$$R_2 \cdot I_2 + R_3 \cdot (I_2 - I_1) = -E_2$$

$$3I_2 + 4(I_2 - I_1) = -5$$

$$-4I_1 + 7I_2 = -5$$

**Paso 3:** Forma matricial:

$$\begin{pmatrix} 6 & -4 \\ -4 & 7 \end{pmatrix} \begin{pmatrix} I_1 \\ I_2 \end{pmatrix} = \begin{pmatrix} 10 \\ -5 \end{pmatrix}$$

**Paso 4:** Resolver:

$$\Delta = 6 \times 7 - (-4)^2 = 42 - 16 = 26$$

$$I_1 = \frac{10 \times 7 - (-5)(-4)}{26} = \frac{70 - 20}{26} = \frac{50}{26} = 1.923\;\text{A}$$

$$I_2 = \frac{6 \times (-5) - (-4) \times 10}{26} = \frac{-30 + 40}{26} = \frac{10}{26} = 0.385\;\text{A}$$

**Corriente por $R_3$**: $I_{R_3} = I_1 - I_2 = 1.923 - 0.385 = 1.538$ A (de arriba a abajo).

$$\boxed{I_1 = 1.923\;\text{A}, \quad I_2 = 0.385\;\text{A}}$$

In [ ]:
# Diagrama P6: 2 mallas
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_title('P6: Análisis de mallas - 2 mallas', fontsize=14, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)

# Malla izquierda
d += elm.SourceV().up().label(r'$E_1=10$ V').reverse()
top_left = d.here
d += elm.Resistor().right().label(r'$R_1=2\;\Omega$')
mid_top = d.here
d += elm.Resistor().down().label(r'$R_3=4\;\Omega$', loc='right')
mid_bot = d.here
d += elm.Line().left().tox(top_left)

# Malla derecha
d.here = mid_top
d += elm.Resistor().right().label(r'$R_2=3\;\Omega$')
d += elm.SourceV().down().label(r'$E_2=5$ V')
d += elm.Line().left().tox(mid_bot)

# Etiquetas de corriente de malla
d += elm.CurrentLabelInline(direction='ccw').at(top_left).label(r'$I_1$')

d.draw()
plt.tight_layout()
plt.show()

### 7.6 Problema 7: 3 mallas con todas las R iguales

**Datos:** 3 mallas, todas las resistencias $R = 1\;\Omega$. $E_g = 10$ V en la malla 1.

#### Ejercicio resuelto: Análisis de mallas P7

**Paso 1:** Con $R = 1\;\Omega$ para todas las resistencias, la diagonal tiene la suma de resistencias en cada malla y los términos cruzados son las compartidas (con signo negativo).

Supongamos que cada malla tiene 3 resistencias y comparte una con cada malla adyacente:

**Paso 2:** Forma matricial:

$$\begin{pmatrix} 3 & -1 & 0 \\ -1 & 3 & -1 \\ 0 & -1 & 3 \end{pmatrix} \begin{pmatrix} I_1 \\ I_2 \\ I_3 \end{pmatrix} = \begin{pmatrix} 10 \\ 0 \\ 0 \end{pmatrix}$$

**Paso 3:** Resolver:

$$\Delta = 3(3 \times 3 - 1) - (-1)(-1 \times 3) = 3 \times 8 - 3 = 21$$

$$I_1 = \frac{10 \times 8}{21} = \frac{80}{21} = 3.810\;\text{A}$$

$$I_2 = \frac{10 \times 3}{21} = \frac{30}{21} = 1.429\;\text{A}$$

$$I_3 = \frac{10 \times 1}{21} = \frac{10}{21} = 0.476\;\text{A}$$

$$\boxed{I_1 = 3.810\;\text{A}, \quad I_2 = 1.429\;\text{A}, \quad I_3 = 0.476\;\text{A}}$$

> **Observación**: Al estar todas las $R$ iguales, la corriente decrece con la distancia a la fuente.

In [ ]:
# Verificación numérica P7
R_mat = np.array([[3, -1, 0],
                   [-1, 3, -1],
                   [0, -1, 3]], dtype=float)
Vs = np.array([10, 0, 0], dtype=float)
I_malla = np.linalg.solve(R_mat, Vs)

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_title('P7: Corrientes de malla (R iguales)', fontsize=14, fontweight='bold')
mallas = [r'$I_1$', r'$I_2$', r'$I_3$']
barras = ax.bar(mallas, I_malla, color=[COLOR_PRINCIPAL, COLOR_RECTA, COLOR_PUNTO],
                width=0.5, edgecolor='black')
for bar, val in zip(barras, I_malla):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.3f} A', ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel(r'Corriente (A)')
ax.set_ylim(0, max(I_malla)*1.3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.7 Problema 8: 3 mallas con fuente de corriente (supermalla)

**Datos:** 3 mallas. $E_g = 20$ V en malla 1, $I_g = 2$ A entre mallas 2 y 3. $R_1 = 5\;\Omega$, $R_2 = 10\;\Omega$, $R_3 = 4\;\Omega$, $R_4 = 8\;\Omega$, $R_5 = 6\;\Omega$.

#### Ejercicio resuelto: Supermalla P8

**Paso 1:** La fuente $I_g$ entre mallas 2 y 3 crea una **supermalla** $\{2, 3\}$.

**Restricción:**
$$I_2 - I_3 = I_g = 2\;\text{A}$$

**Paso 2:** KVL malla 1:

$$R_1 \cdot I_1 + R_2 \cdot (I_1 - I_2) = E_g$$

$$5I_1 + 10(I_1 - I_2) = 20$$

$$15I_1 - 10I_2 = 20$$

**KVL supermalla** $\{2, 3\}$ (recorriendo el contorno exterior, saltando $I_g$):

$$R_2(I_2 - I_1) + R_3 \cdot I_2 + R_5 \cdot I_3 + R_4 \cdot I_3 = 0$$

$$10(I_2 - I_1) + 4I_2 + 6I_3 + 8I_3 = 0$$

$$-10I_1 + 14I_2 + 14I_3 = 0$$

**Paso 3:** Sistema de 3 ecuaciones:

$$\begin{cases} 15I_1 - 10I_2 = 20 \\ -10I_1 + 14I_2 + 14I_3 = 0 \\ I_2 - I_3 = 2 \end{cases}$$

De la tercera: $I_3 = I_2 - 2$. Sustituyendo:

$$15I_1 - 10I_2 = 20$$
$$-10I_1 + 14I_2 + 14(I_2 - 2) = 0 \implies -10I_1 + 28I_2 = 28$$

De la primera: $I_1 = \frac{20 + 10I_2}{15}$. Sustituyendo:

$$-10 \cdot \frac{20 + 10I_2}{15} + 28I_2 = 28$$

$$-\frac{200 + 100I_2}{15} + 28I_2 = 28$$

$$-200 - 100I_2 + 420I_2 = 420$$

$$320I_2 = 620 \implies I_2 = 1.9375\;\text{A}$$

$$I_3 = 1.9375 - 2 = -0.0625\;\text{A}$$

$$I_1 = \frac{20 + 10(1.9375)}{15} = \frac{39.375}{15} = 2.625\;\text{A}$$

$$\boxed{I_1 = 2.625\;\text{A}, \quad I_2 = 1.938\;\text{A}, \quad I_3 = -0.063\;\text{A}}$$

### 7.8 Problema 10: Fuentes dependientes

**Datos:** Circuito con 2 nudos. $R_1 = 2\;\Omega$ (de $A$ a ref), $R_2 = 4\;\Omega$ (de $A$ a $B$), $R_3 = 8\;\Omega$ (de $B$ a ref). Fuente dependiente de corriente: $I_d = 3V_A$ (controlada por $V_A$), entra al nudo $B$. Fuente independiente: $I_g = 5$ A entra a nudo $A$.

#### Ejercicio resuelto: Fuentes dependientes P10

**Paso 1:** Ecuaciones de nudos normales, tratando $I_d$ como fuente.

**Nudo A:**
$$\frac{V_A}{R_1} + \frac{V_A - V_B}{R_2} = I_g$$

$$\frac{V_A}{2} + \frac{V_A - V_B}{4} = 5$$

$$\frac{3}{4}V_A - \frac{1}{4}V_B = 5 \quad \ldots (1)$$

**Nudo B:**
$$\frac{V_B - V_A}{R_2} + \frac{V_B}{R_3} = -I_d = -3V_A$$

$$\frac{V_B - V_A}{4} + \frac{V_B}{8} = -3V_A$$

$$-\frac{1}{4}V_A + \frac{3}{8}V_B + 3V_A = 0$$

$$\frac{11}{4}V_A + \frac{3}{8}V_B = 0 \quad \ldots (2)$$

**Paso 2:** Observar que la matriz **no es simétrica** por la fuente dependiente:

$$\begin{pmatrix} 3/4 & -1/4 \\ 11/4 & 3/8 \end{pmatrix} \begin{pmatrix} V_A \\ V_B \end{pmatrix} = \begin{pmatrix} 5 \\ 0 \end{pmatrix}$$

**Paso 3:** De (2): $V_B = -\frac{11/4}{3/8}V_A = -\frac{22}{3}V_A$.

Sustituyendo en (1):

$$\frac{3}{4}V_A - \frac{1}{4}\left(-\frac{22}{3}V_A\right) = 5$$

$$\frac{3}{4}V_A + \frac{22}{12}V_A = 5$$

$$\frac{9 + 22}{12}V_A = 5 \implies \frac{31}{12}V_A = 5$$

$$V_A = \frac{60}{31} = 1.935\;\text{V}$$

$$V_B = -\frac{22}{3} \times 1.935 = -14.19\;\text{V}$$

$$\boxed{V_A = 1.94\;\text{V}, \quad V_B = -14.19\;\text{V}}$$

> **Nota**: La asimetría de la matriz es característica de las fuentes dependientes. No se puede usar inspección directa.

In [ ]:
# Verificación numérica P10 con fuente dependiente
G_dep = np.array([[3/4, -1/4],
                   [11/4, 3/8]], dtype=float)
Is_dep = np.array([5, 0], dtype=float)
V_dep = np.linalg.solve(G_dep, Is_dep)

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_title('P10: Tensiones con fuente dependiente', fontsize=14, fontweight='bold')
nudos = [r'$V_A$', r'$V_B$']
colors = [COLOR_PRINCIPAL, COLOR_RECTA]
barras = ax.bar(nudos, V_dep, color=colors, width=0.4, edgecolor='black')
for bar, val in zip(barras, V_dep):
    ypos = bar.get_height() + 0.2 if val >= 0 else bar.get_height() - 0.8
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'{val:.2f} V', ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel(r'Tensión (V)')
ax.axhline(y=0, color='black', lw=0.8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 8. Catálogo completo de ejercicios: todos los patrones

| # | Tipo | Método | Ecuación clave | Dificultad |
|---|------|--------|----------------|------------|
| 1 | Análisis nodal puro | Nudos | $\mathbf{G}\mathbf{V} = \mathbf{I}_s$ | Baja |
| 2 | Nodal con fuente de tensión a ref | Nudos | $V_k = E_g$ (directo) | Baja |
| 3 | Nodal con supernudo | Nudos | $V_k - V_j = E_g$ + KCL supernudo | Media |
| 4 | Análisis de mallas puro | Mallas | $\mathbf{R}\mathbf{I} = \mathbf{V}_s$ | Baja |
| 5 | Mallas con fuente de corriente exterior | Mallas | $I_k = \pm I_g$ (directo) | Baja |
| 6 | Mallas con supermalla | Mallas | $I_k - I_j = I_g$ + KVL supermalla | Media |
| 7 | Nodal con fuente dependiente | Nudos | $\mathbf{G}$ no simétrica | Alta |
| 8 | Mallas con fuente dependiente | Mallas | $\mathbf{R}$ no simétrica | Alta |
| 9 | Elegir nudos vs mallas | Ambos | Decidir método óptimo | Media |
| 10 | Circuito mixto (tensión + corriente) | Ambos | Combinación de técnicas | Alta |

### 8.1 Tipo 1: Análisis nodal puro (solo resistencias y fuentes de corriente)

**Identificación**: Solo hay resistencias y fuentes de corriente independientes. No hay fuentes de tensión.

$$\boxed{\mathbf{G} \cdot \mathbf{V} = \mathbf{I}_s}$$

**Construcción por inspección:**
- $G_{kk}$ = suma de conductancias conectadas al nudo $k$
- $G_{kj}$ = $-$(conductancia entre nudos $k$ y $j$)
- $I_{s,k}$ = suma de fuentes de corriente que entran a nudo $k$ (positivas si entran)

**Ventaja**: La matriz es simétrica $\to$ se puede verificar rápidamente.

**Error frecuente**: Olvidar que las conductancias entre un nudo y la referencia también van en la diagonal.

#### Ejercicio resuelto: Tipo 1

**Datos:** 3 nudos ($A$, $B$, ref). $R_1 = 5\;\Omega$ ($A$ a ref), $R_2 = 10\;\Omega$ ($A$ a $B$), $R_3 = 20\;\Omega$ ($B$ a ref). $I_g = 2$ A entra a $A$.

$$\begin{pmatrix} \frac{1}{5}+\frac{1}{10} & -\frac{1}{10} \\ -\frac{1}{10} & \frac{1}{10}+\frac{1}{20} \end{pmatrix} \begin{pmatrix} V_A \\ V_B \end{pmatrix} = \begin{pmatrix} 2 \\ 0 \end{pmatrix}$$

$$\begin{pmatrix} 0.3 & -0.1 \\ -0.1 & 0.15 \end{pmatrix} \begin{pmatrix} V_A \\ V_B \end{pmatrix} = \begin{pmatrix} 2 \\ 0 \end{pmatrix}$$

$$\Delta = 0.3 \times 0.15 - 0.01 = 0.035$$

$$V_A = \frac{2 \times 0.15}{0.035} = 8.57\;\text{V}, \quad V_B = \frac{0.1 \times 2}{0.035} = 5.71\;\text{V}$$

$$\boxed{V_A = 8.57\;\text{V}, \quad V_B = 5.71\;\text{V}}$$

### 8.2 Tipo 2: Nodal con fuente de tensión conectada a referencia

**Identificación**: Hay una fuente de tensión entre un nudo y la referencia.

**Truco**: La tensión de ese nudo es directamente $V_k = E_g$. Se elimina una incógnita del sistema.

$$\boxed{V_k = E_g \quad \text{(se conoce directamente)}}$$

**Procedimiento:**
1. Fijar $V_k = E_g$.
2. Escribir KCL en los demás nudos sustituyendo $V_k$.
3. Resolver sistema reducido de $(n-2)$ ecuaciones.

#### Ejercicio resuelto: Tipo 2

**Datos:** $E_g = 24$ V fija $V_A = 24$ V. $R_1 = 6\;\Omega$ ($A$ a $B$), $R_2 = 12\;\Omega$ ($B$ a ref), $R_3 = 4\;\Omega$ ($B$ a ref).

KCL en $B$:
$$\frac{V_B - 24}{6} + \frac{V_B}{12} + \frac{V_B}{4} = 0$$

$$V_B\left(\frac{1}{6} + \frac{1}{12} + \frac{1}{4}\right) = \frac{24}{6} = 4$$

$$V_B \cdot 0.5 = 4 \implies \boxed{V_B = 8\;\text{V}}$$

### 8.3 Tipo 3: Nodal con supernudo

**Identificación**: Hay una fuente de tensión entre dos nudos (ninguno es referencia).

$$\boxed{V_k - V_j = E_g \quad + \quad \text{KCL del supernudo (superficie que engloba } k \text{ y } j)}$$

**Procedimiento:**
1. Escribir la restricción $V_k - V_j = E_g$.
2. Dibujar una superficie cerrada que englobe ambos nudos.
3. Escribir KCL para esa superficie (sumando todas las corrientes que cruzan la frontera).
4. Escribir KCL normalmente en los demás nudos.

**Error frecuente**: Escribir KCL en los nudos $k$ y $j$ por separado (no se puede porque no conocemos la corriente por la fuente de tensión).

#### Ejercicio resuelto: Tipo 3

**Datos:** $E_g = 10$ V entre $A$ y $B$ ($V_A - V_B = 10$). $R_1 = 2\;\Omega$ ($A$ a ref), $R_2 = 5\;\Omega$ ($B$ a ref), $I_g = 4$ A entra al supernudo por $A$.

**Restricción:** $V_A - V_B = 10$

**KCL supernudo:** $\frac{V_A}{2} + \frac{V_B}{5} = 4$

Sustituimos $V_A = V_B + 10$:

$$\frac{V_B + 10}{2} + \frac{V_B}{5} = 4$$

$$V_B\left(\frac{1}{2} + \frac{1}{5}\right) = 4 - 5 = -1$$

$$V_B \cdot 0.7 = -1 \implies V_B = -1.43\;\text{V}$$

$$\boxed{V_A = 8.57\;\text{V}, \quad V_B = -1.43\;\text{V}}$$

### 8.4 Tipo 4: Análisis de mallas puro (solo resistencias y fuentes de tensión)

**Identificación**: Solo hay resistencias y fuentes de tensión independientes. No hay fuentes de corriente.

$$\boxed{\mathbf{R} \cdot \mathbf{I} = \mathbf{V}_s}$$

**Construcción por inspección:**
- $R_{kk}$ = suma de resistencias en la malla $k$
- $R_{kj}$ = $-$(resistencia compartida entre mallas $k$ y $j$)
- $V_{s,k}$ = suma algebraica de fuentes de tensión en malla $k$ (positiva si sube en sentido de $I_k$)

#### Ejercicio resuelto: Tipo 4

**Datos:** 2 mallas. $E = 15$ V, $R_1 = 3\;\Omega$, $R_2 = 6\;\Omega$ (compartida), $R_3 = 9\;\Omega$.

$$\begin{pmatrix} 3+6 & -6 \\ -6 & 6+9 \end{pmatrix} \begin{pmatrix} I_1 \\ I_2 \end{pmatrix} = \begin{pmatrix} 15 \\ 0 \end{pmatrix}$$

$$\Delta = 9 \times 15 - 36 = 99$$

$$I_1 = \frac{15 \times 15}{99} = 2.273\;\text{A}, \quad I_2 = \frac{6 \times 15}{99} = 0.909\;\text{A}$$

$$\boxed{I_1 = 2.273\;\text{A}, \quad I_2 = 0.909\;\text{A}}$$

### 8.5 Tipo 5: Mallas con fuente de corriente en rama exterior

**Identificación**: Hay una fuente de corriente en una rama que pertenece a una sola malla.

$$\boxed{I_k = \pm I_g \quad \text{(directamente conocida)}}$$

**Procedimiento:**
1. La corriente de esa malla es conocida directamente.
2. Sustituir en las demás ecuaciones de malla.
3. Resolver el sistema reducido.

#### Ejercicio resuelto: Tipo 5

**Datos:** 2 mallas. $I_g = 3$ A en rama exterior de malla 1 $\to$ $I_1 = 3$ A. $E = 12$ V, $R_1 = 4\;\Omega$ (compartida), $R_2 = 6\;\Omega$.

KVL malla 2: $R_1(I_2 - I_1) + R_2 \cdot I_2 = -E$

$$4(I_2 - 3) + 6I_2 = -12$$

$$10I_2 = -12 + 12 = 0 \implies \boxed{I_2 = 0\;\text{A}}$$

### 8.6 Tipo 6: Mallas con supermalla

**Identificación**: Hay una fuente de corriente en una rama compartida por dos mallas.

$$\boxed{I_k - I_j = \pm I_g \quad + \quad \text{KVL de la supermalla (contorno exterior)}}$$

**Procedimiento:**
1. Escribir la restricción $I_k - I_j = \pm I_g$.
2. Combinar las dos mallas en una supermalla.
3. Escribir KVL recorriendo el contorno exterior (saltando la fuente de corriente).

#### Ejercicio resuelto: Tipo 6

**Datos:** 2 mallas. $I_g = 5$ A entre mallas 1 y 2 ($I_1 - I_2 = 5$). $E = 20$ V, $R_1 = 4\;\Omega$ (malla 1 exterior), $R_2 = 6\;\Omega$ (malla 2 exterior).

**Restricción:** $I_1 - I_2 = 5$

**KVL supermalla** (contorno exterior):
$$R_1 \cdot I_1 + R_2 \cdot I_2 = E$$
$$4I_1 + 6I_2 = 20$$

Sustituyendo $I_1 = I_2 + 5$:
$$4(I_2 + 5) + 6I_2 = 20 \implies 10I_2 = 0 \implies I_2 = 0\;\text{A}$$

$$\boxed{I_1 = 5\;\text{A}, \quad I_2 = 0\;\text{A}}$$

### 8.7 Tipo 7: Nodal con fuente dependiente

**Identificación**: Hay una fuente dependiente (controlada). La variable de control debe expresarse en función de las tensiones de nudo.

**Procedimiento:**
1. Escribir KCL como siempre, incluyendo la fuente dependiente como si fuera independiente.
2. Expresar la variable de control en función de $V_k$.
3. Sustituir y reagrupar.

> **Clave**: La matriz $\mathbf{G}$ pierde la simetría.

#### Ejercicio resuelto: Tipo 7

**Datos:** 2 nudos. $R_1 = 4\;\Omega$ ($A$ a ref), $R_2 = 2\;\Omega$ ($A$ a $B$), $R_3 = 8\;\Omega$ ($B$ a ref). Fuente dependiente: $I_d = 2V_{AB}$ (donde $V_{AB} = V_A - V_B$), entra a $B$. $I_g = 6$ A entra a $A$.

**Nudo A:** $\frac{V_A}{4} + \frac{V_A - V_B}{2} = 6 \implies \frac{3}{4}V_A - \frac{1}{2}V_B = 6$

**Nudo B:** $\frac{V_B - V_A}{2} + \frac{V_B}{8} + 2(V_A - V_B) = 0$

$$-\frac{1}{2}V_A + \frac{1}{8}V_B + 2V_A - 2V_B = 0$$

$$\frac{3}{2}V_A - \frac{15}{8}V_B = 0 \implies V_B = \frac{4}{5}V_A$$

Sustituyendo en nudo A:
$$\frac{3}{4}V_A - \frac{1}{2} \cdot \frac{4}{5}V_A = 6$$

$$V_A\left(\frac{3}{4} - \frac{2}{5}\right) = 6 \implies V_A \cdot \frac{7}{20} = 6$$

$$\boxed{V_A = 17.14\;\text{V}, \quad V_B = 13.71\;\text{V}}$$

### 8.8 Tipo 8: Mallas con fuente dependiente

**Identificación**: Hay una fuente dependiente. La variable de control debe expresarse en función de las corrientes de malla.

**Procedimiento:**
1. Escribir KVL como siempre, incluyendo la fuente dependiente.
2. Expresar la variable de control en función de $I_k$.
3. Sustituir y reagrupar.

#### Ejercicio resuelto: Tipo 8

**Datos:** 2 mallas. $R_1 = 3\;\Omega$, $R_2 = 6\;\Omega$ (compartida), $R_3 = 9\;\Omega$. Fuente dependiente $E_d = 4I_x$ donde $I_x = I_1$ (corriente por $R_1$). $E = 18$ V en malla 1.

**KVL malla 1:** $R_1 I_1 + R_2(I_1 - I_2) = E + E_d = 18 + 4I_1$

$$3I_1 + 6(I_1 - I_2) = 18 + 4I_1 \implies 5I_1 - 6I_2 = 18$$

**KVL malla 2:** $R_2(I_2 - I_1) + R_3 I_2 = 0$

$$-6I_1 + 15I_2 = 0 \implies I_1 = 2.5I_2$$

$$5(2.5I_2) - 6I_2 = 18 \implies 6.5I_2 = 18 \implies I_2 = 2.769\;\text{A}$$

$$\boxed{I_1 = 6.923\;\text{A}, \quad I_2 = 2.769\;\text{A}}$$

### 8.9 Tipo 9: Elegir entre nudos y mallas

**Regla práctica** para elegir el método más eficiente:

| Situación | Método recomendado | Razón |
|-----------|--------------------|-------|
| Más fuentes de corriente que de tensión | **Nudos** | Las fuentes de corriente van directamente al vector $\mathbf{I}_s$ |
| Más fuentes de tensión que de corriente | **Mallas** | Las fuentes de tensión van directamente al vector $\mathbf{V}_s$ |
| Pocos nudos, muchas mallas | **Nudos** | Menos ecuaciones ($n-1 < c-n+1$) |
| Pocas mallas, muchos nudos | **Mallas** | Menos ecuaciones |
| Fuente de tensión a referencia | **Nudos** | Elimina una incógnita directamente |
| Fuente de corriente en rama exterior | **Mallas** | Elimina una incógnita directamente |
| Circuito no plano | **Nudos** | No se pueden definir mallas en circuitos no planos |

> **Truco para el examen**: Cuenta el número de ecuaciones que tendrías con cada método. El que dé menos ecuaciones es el mejor.

#### Ejercicio resuelto: Tipo 9

**Datos:** Circuito con $n = 4$ nudos, $c = 6$ componentes, 2 fuentes de corriente y 1 fuente de tensión conectada a referencia.

- **Nudos**: $n - 1 = 3$ ecuaciones, pero la fuente de tensión a ref elimina 1 $\to$ **2 ecuaciones**.
- **Mallas**: $c - n + 1 = 3$ ecuaciones, y hay 2 fuentes de corriente que pueden complicar.

$\to$ **Método de nudos** es claramente mejor (2 ecuaciones vs 3, y las fuentes de corriente son naturales en nudos).

### 8.10 Tipo 10: Circuito mixto (fuentes de tensión y corriente simultáneas)

**Identificación**: El circuito tiene tanto fuentes de tensión como de corriente. Requiere supernudos/supermallas o elegir el método adecuado.

**Estrategia:**
1. Contar ecuaciones con cada método.
2. Ver si alguna fuente simplifica directamente una incógnita.
3. Aplicar el método elegido con las técnicas de supernudo/supermalla según corresponda.

#### Ejercicio resuelto: Tipo 10

**Datos:** 3 nudos. $E_g = 8$ V (de ref a $A$), $I_g = 3$ A (entra a $B$), $R_1 = 2\;\Omega$ ($A$ a $B$), $R_2 = 4\;\Omega$ ($B$ a ref).

**Método de nudos** (mejor opción porque $E_g$ fija $V_A$ directamente):

$V_A = 8$ V (directamente de $E_g$).

KCL en $B$:
$$\frac{V_B - V_A}{R_1} + \frac{V_B}{R_2} = -I_g$$

$$\frac{V_B - 8}{2} + \frac{V_B}{4} = -3$$

$$V_B\left(\frac{1}{2} + \frac{1}{4}\right) = -3 + 4 = 1$$

$$\boxed{V_A = 8\;\text{V}, \quad V_B = \frac{4}{3} = 1.33\;\text{V}}$$

---

## 9. Resumen y tabla de fórmulas clave

| Concepto | Fórmula / Regla |
|----------|-----------------|
| **Incógnitas** | $2c$ (tensión y corriente por cada componente) |
| **Ecuaciones** | $c$ (componente) + $(n-1)$ (KCL) + $(c-n+1)$ (KVL) $= 2c$ |
| **Ecuaciones de nudos** | $\boxed{\mathbf{G} \cdot \mathbf{V} = \mathbf{I}_s}$ |
| $G_{kk}$ | Suma de conductancias conectadas a nudo $k$ |
| $G_{kj}$ | $-$(conductancia entre nudos $k$ y $j$) |
| **Ecuaciones de mallas** | $\boxed{\mathbf{R} \cdot \mathbf{I} = \mathbf{V}_s}$ |
| $R_{kk}$ | Suma de resistencias en malla $k$ |
| $R_{kj}$ | $-$(resistencia compartida entre mallas $k$ y $j$) |
| **Supernudo** | $V_k - V_j = E_g$ + KCL del supernudo |
| **Supermalla** | $I_k - I_j = I_g$ + KVL de la supermalla |
| **Fuentes dependientes** | Expresar variable de control en función de incógnitas; matrices no simétricas |
| **Elegir método** | Nudos si hay más fuentes de corriente; mallas si hay más fuentes de tensión |

### Errores frecuentes en examen

1. **Olvidar el signo** de $G_{kj}$ (siempre negativo si no hay fuentes dependientes).
2. **No incluir** la conductancia entre nudo y referencia en $G_{kk}$.
3. **Confundir sentido** de la fuente de corriente al montar $\mathbf{I}_s$.
4. **Escribir KCL en nudos del supernudo** por separado (no se puede).
5. **No expresar** la variable de control de la fuente dependiente en función de las incógnitas.
6. **Asumir simetría** de la matriz cuando hay fuentes dependientes.